# 01 - Simulación de la señal rPPG (verdad conocida)

**Qué hace este notebook:** genera una señal de pulso sintética, simula cómo la vería una cámara (canales RGB con ruido, movimiento, parpadeo y cuantización a 8 bits) y comprueba que el método de estimación recupera el BPM real.

**Por qué simular:** aquí SÍ conocemos el BPM verdadero, así que podemos medir el error con exactitud (con un video real no).

**Conceptos de Teoría de Señales:** muestreo (fs, Ts), señal discreta, dominio temporal, caracterización (media, amplitud, desviación), ruido, aliasing, análisis en frecuencia (FFT), normalización.

> Uso educativo. Nada de esto es una medición clínica ni diagnóstica.

In [ ]:
import os, sys, subprocess

# --- Configuración (EDITA REPO_URL con el enlace de TU repositorio de GitHub) ---
REPO_URL = "https://github.com/sh-011188/Proyecto_Senales.git"
REPO_DIR = "Proyecto_Senales"   # nombre de la carpeta del repositorio

IN_COLAB = "google.colab" in sys.modules
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                           # ejecución local: subir a la raíz del repo
elif not os.path.isdir("src"):
    if "TU_USUARIO" in REPO_URL:
        raise SystemExit("Edita REPO_URL con el enlace de tu repositorio y vuelve a ejecutar.")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

sys.path.insert(0, os.getcwd())
print("Directorio de trabajo:", os.getcwd())


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.simulation import simulate_rgb
from src.analysis import describe_signal, compute_spectrum, estimate_bpm_from_spectrum
from src.preprocessing import normalize_by_mean, detrend_linear
from src.bpm import estimate_bpm
from src import plots
plt.rcParams["figure.dpi"] = 100

## 1. Generación de la señal (adquisición simulada)

La señal se genera computacionalmente (no viene de un sensor). Parámetros principales:
- `fs = 30 Hz` (como un video de 30 fps) -> `Ts = 1/fs`.
- `f0_mean = 1.2 Hz` -> 72 BPM. La frecuencia varía un poco con la respiración (RSA).
- Ruido de sensor, movimiento (multiplicativo, de baja frecuencia), parpadeo de luz y cuantización a 8 bits.

In [ ]:
sim = simulate_rgb(fs=30.0, duration=30.0, f0_mean=1.2, motion_std=0.01, seed=42)

print(f"fs = {sim['fs']} Hz  |  Ts = {1/sim['fs']*1000:.2f} ms")
print(f"Muestras: {len(sim['t'])}  |  duración: {sim['t'][-1] + 1/sim['fs']:.1f} s")
print(f"BPM real (media): {sim['bpm_true_mean']:.2f}")
print(f"Parpadeo de 100 Hz muestreado a {sim['fs']} Hz aparece (aliasing) en: {sim['flicker_alias_hz']:.1f} Hz")
plots.plot_simulation_overview(sim);

### ✍️ Interpretación (escríbela con tus palabras)
- ¿Qué forma tiene el pulso? ¿Es una senoide pura? ¿Por qué se modeló con armónicos?
- ¿Por qué la frecuencia cardíaca real (panel inferior) no es una línea recta?
- ¿Cuántos latidos aparecen en la ventana de 8 s y cuál es el periodo aproximado?

*(Escribe aquí tu respuesta.)*

In [ ]:
plots.plot_rgb_time(sim["t"], sim["rgb"], "Series RGB simuladas (ruido + movimiento + cuantización)");

# Señal limpia vs. con ruido (canal G), primeros 10 s
k = int(10 * sim["fs"])
plt.figure(figsize=(10, 3.5))
plt.plot(sim["t"][:k], sim["rgb_clean"][:k, 1], label="G limpio (solo pulso)", lw=1.5)
plt.plot(sim["t"][:k], sim["rgb"][:k, 1], label="G observado (con degradaciones)", lw=0.8, alpha=0.8)
plt.xlabel("Tiempo (s)"); plt.ylabel("Nivel (0-255)"); plt.title("Efecto del ruido y el movimiento en el canal verde")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
tabla = pd.DataFrame({c: describe_signal(sim["rgb"][:, i], sim["fs"]) for i, c in enumerate("RGB")}).T
tabla[["n_muestras", "fs_hz", "duracion_s", "minimo", "maximo", "amplitud_pico_a_pico", "media", "desv_estandar"]].round(3)

### ✍️ Interpretación
- ¿Qué tan grande es la variación del pulso comparada con el nivel medio (en %)? ¿Por qué es tan pequeña?
- ¿Qué canal varía más y por qué? ¿Qué efecto tiene la cuantización a 8 bits?
- ¿Qué observas en la señal con ruido que no está en la limpia (deriva lenta, ruido rápido)?

*(Escribe aquí tu respuesta.)*

## 2. Espectro de la señal cruda y aliasing

La FFT descompone la señal en frecuencias. Eje X: frecuencia en Hz (de 0 a Nyquist = fs/2 = 15 Hz). Eje Y: amplitud. Se calcula sobre el canal G sin procesar (se quita solo la media).

In [ ]:
freqs, mag = compute_spectrum(sim["rgb"][:, 1], sim["fs"])
est = estimate_bpm_from_spectrum(freqs, mag)
plots.plot_spectrum(freqs, mag, est, title="Espectro del canal G crudo (0 a Nyquist)",
                    true_bpm=sim["bpm_true_mean"], fmax_hz=sim["fs"]/2);
print(f"Resolución espectral real: df = fs/N = {sim['fs']/len(sim['t']):.3f} Hz = {60*sim['fs']/len(sim['t']):.1f} BPM")

### ✍️ Interpretación
- Identifica en el espectro: (a) el pico del pulso, (b) la energía de baja frecuencia (movimiento/deriva), (c) el pico del parpadeo.
- **Aliasing:** el parpadeo físico es de 100 Hz, pero con fs = 30 Hz aparece en 10 Hz. Comprueba: `|100 - 30*round(100/30)| = 10`. ¿Por qué ocurre y qué debería cumplirse (Nyquist) para evitarlo?
- Con el parpadeo a 60 Hz y fs = 30 Hz, la senoide se muestrea siempre en sus cruces por cero y desaparece. ¿Por qué?

*(Escribe aquí tu respuesta.)*

## 3. Primera etapa de procesamiento: normalización y detrending

Se pasa a variación relativa (`x/media - 1`) y se quita la tendencia lineal.

In [ ]:
g = sim["rgb"][:, 1]
plots.plot_raw_vs_processed(sim["t"], g, detrend_linear(normalize_by_mean(g)));

### ✍️ Interpretación
- ¿Qué cambió entre la señal cruda y la procesada (escala, deriva)? ¿Por qué conviene trabajar con variación relativa?

*(Escribe aquí tu respuesta.)*

## 4. Estimación del BPM y comparación con la verdad conocida

Dos métodos: **POS** (combina R, G, B) y **canal verde** (método base). Se compara el pico espectral con el BPM real.

In [ ]:
res = {m: estimate_bpm(sim["rgb"], sim["fs"], m) for m in ("pos", "green")}
pd.DataFrame({
    "BPM estimado": {m: round(r["bpm"], 2) for m, r in res.items()},
    "Error (BPM)": {m: round(r["bpm"] - sim["bpm_true_mean"], 2) for m, r in res.items()},
    "Calidad (heurística)": {m: round(r["quality"], 2) for m, r in res.items()},
}).assign(**{"BPM real": round(sim["bpm_true_mean"], 2)})

In [ ]:
plots.plot_spectrum(res["pos"]["freqs"], res["pos"]["magnitude"], res["pos"],
                    title="Espectro de la señal de pulso (POS)", true_bpm=sim["bpm_true_mean"]);

### ✍️ Interpretación
- ¿Qué tan cerca quedó la estimación del valor real? ¿Cuál es la incertidumbre mínima que impone la resolución espectral?
- ¿Por qué se busca el pico solo dentro de 0.7 a 3 Hz?

*(Escribe aquí tu respuesta.)*

## 5. Experimento opcional (tema de la entrega final): ¿cuándo ayuda POS?

Se repite la simulación con 20 semillas aleatorias y distintos niveles de movimiento, y se mide el error absoluto medio (MAE).

**Ojo:** el movimiento de esta simulación es multiplicativo y casi común a los tres canales, que es justo el supuesto en el que POS se apoya. Que POS gane aquí es en parte consecuencia del modelo, no prueba de que gane en video real.

In [ ]:
filas = []
for mstd in (0.01, 0.03, 0.05, 0.10):
    ep, eg = [], []
    for seed in range(20):
        s = simulate_rgb(motion_std=mstd, motion_theta=2.0, seed=seed)
        ep.append(abs(estimate_bpm(s["rgb"], s["fs"], "pos")["bpm"] - s["bpm_true_mean"]))
        eg.append(abs(estimate_bpm(s["rgb"], s["fs"], "green")["bpm"] - s["bpm_true_mean"]))
    filas.append({"movimiento (std relativa)": mstd, "MAE POS (BPM)": np.mean(ep), "MAE verde (BPM)": np.mean(eg)})
pd.DataFrame(filas).round(2)

## Conclusiones (escríbelas tú)
- Qué funciona ya y qué falta para la entrega final.
- Limitaciones de esta simulación.

*(Escribe aquí.)*